# マルチモーダル処理用のカスタムツール

`@tool`デコレータを使用してカスタムツールでエージェントを拡張します。このノートブックでは、カスタムツール実装を使用して画像、動画、ドキュメントを含むマルチモーダルコンテンツを処理する方法を説明します。

## 学習内容

- `@tool`デコレータでカスタムツールを作成する
- 画像、動画、ドキュメントを処理する
- エージェントワークフローでマルチモーダルコンテンツを処理する
- ツールでエラーハンドリングとバリデーションを実装する

## 前提条件

- [ノートブック01: Hello World](01-hello-world-strands-agents.ipynb)を完了していること
- Python関数とデコレータの理解
- サンプルメディアファイル（`data-sample/`ディレクトリに提供）

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool
from datetime import datetime

print("✅ Imports successful!")

## シンプルなツールを作成する

シンプルな計算機ツールを作成しましょう（@toolデコレータを使用）：

In [ ]:
@tool
def calculator(operation: str, a: float, b: float) -> float:
    """基本的な数学演算を実行します。
    
    Args:
        operation: 実行する演算（add, subtract, multiply, divide）
        a: 最初の数値
        b: 2番目の数値
    
    Returns:
        演算の結果
    """
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    return operations.get(operation, "Invalid operation")

print("✅ 計算機ツールを作成しました！")

## エージェントでツールを使用する

まず、エージェントが使用するモデルインスタンスを作成しましょう

In [ ]:
# Bedrockモデルを設定する
session = boto3.Session(region_name='us-east-1')
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    boto_session=session
)

計算機を使用できるエージェントを作成しましょう：

In [ ]:
# 計算機ツールでエージェントを作成する
math_agent = Agent(
    model=bedrock_model,
    tools=[calculator],
    system_prompt="あなたは親切な数学アシスタントです。計算機ツールを使用して計算を実行してください。"
)

print("✅ 計算機ツールで数学エージェントを作成しました！")

In [ ]:
# 計算機ツールをテストする
response = math_agent("156に23を掛けると何ですか？")
print(response)

StrandsのAgentResult型を確認しましょう：

In [ ]:
# AgentResultオブジェクトを確認する
print(f"メッセージ: {response.message}")
print("-" * 100 + "\n")

print(f"メトリクス: {response.metrics}")
print("-" * 100 + "\n")

print(f"状態: {response.state}")
print("-" * 100 + "\n")

print(f"停止理由: {response.stop_reason}")
print("-" * 100 + "\n")

## より複雑なツールを作成する

現在時刻情報を取得するツールを作成しましょう：

In [ ]:
@tool
def get_current_time(timezone: str = "UTC") -> str:
    """現在の日付と時刻を取得します。
    
    Args:
        timezone: タイムゾーン（現在はUTCのみサポート）
    
    Returns:
        文字列としての現在の日付と時刻
    """
    now = datetime.now()
    return f"現在時刻 ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"

print("✅ 時刻ツールを作成しました！")

In [ ]:
# 複数のツールでエージェントを作成する
assistant = Agent(
    model=bedrock_model,
    tools=[calculator, get_current_time],
    system_prompt="あなたは計算機と時刻ツールにアクセスできる親切なアシスタントです。"
)

response = assistant("今何時ですか？また、50に75を足すと何ですか？")
print(response)

## 組み込みツールを使用する

Strands Agentsには、一般的なタスク用の事前構築されたツールが含まれています：

In [ ]:
from strands_tools import image_reader, file_read 
# video_readerツール構造の例
# （これはvideo_reader.pyに既に実装されています）

from video_reader_local import video_reader_local

# 組み込みツールでエージェントを作成する
multimodal_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read,video_reader_local],
    system_prompt="""あなたは次のことができるマルチモーダルアシスタントです：
    - 画像を読み取り、分析する
    - ドキュメント（PDF、CSV、DOCXなど）を処理する
    - 複雑なタスクに高度な推論を使用する。
    - 動画を分析し、詳細な洞察を提供する。
    """
)

print("✅ 組み込みツールでマルチモーダルエージェントを作成しました！")

`agent.tool_name`でエージェントに読み込まれているツールを確認でき、`agent.tool_config`にはツールの説明と入力パラメータを含むツールのJSON表現も含まれています

In [ ]:
print(multimodal_agent.tool_names)

print(multimodal_agent.tool_registry.get_all_tools_config())

In [ ]:
# 例1: 画像分析
print("=== 📸 画像分析 ===")
image_result = multimodal_agent("画像data-sample/diagram.jpgを詳細に分析し、観察したすべてを説明してください")
# print(image_result)
print("\n" + "="*80 + "\n")

In [ ]:
# 例2: ドキュメント分析（PDFドキュメントがある場合）
print("=== 📄 ドキュメント分析 ===")
doc_result = multimodal_agent("ドキュメントdata-sample/Welcome-Strands-Agents-SDK.pdfの内容をJSONとして要約してください")
# print(doc_result)

In [ ]:
# 例2: 動画分析
print("=== 🎬 動画分析 ===")
video_result = multimodal_agent("動画data-sample/climbing-video.mp4を分析し、観察したアクションとシーンを詳細に説明してください")
print(video_result)
print("\n" + "="*80 + "\n")

In [ ]:
# AgentResultオブジェクトを確認する
print(f"メッセージ: {video_result.message}")
print("-" * 100 + "\n")

print(f"メトリクス: {video_result.metrics}")
print("-" * 100 + "\n")

print(f"状態: {video_result.state}")
print("-" * 100 + "\n")

print(f"停止理由: {video_result.stop_reason}")
print("-" * 100 + "\n")

## ツールの直接使用

エージェントからツールを直接呼び出すこともできます：

In [ ]:
print(multimodal_agent.tool_names)

In [ ]:
# 例4. ツールの直接使用
video_analysis = multimodal_agent.tool.video_reader_local(
     video_path="data-sample/climbing-video.mp4", 
     text_prompt="この動画の主な要素は何ですか？"
)

In [ ]:
print(video_analysis)

### 追加サンプル
より大きな動画用にAWS S3バケットを使用する動画リーダーを使用するエージェント。

そのためには、バケット環境変数を追加する必要があります

```bash
export VIDEO_READER_S3_BUCKET = "YOU-BUCKET-NAME"
```

In [ ]:
from strands_tools import image_reader, file_read 
# video_readerツール構造の例
# （これはvideo_reader.pyに既に実装されています）

from video_reader import video_reader


# 組み込みツールでエージェントを作成する
multimodal_agent = Agent(
    model=bedrock_model,
    tools=[image_reader, file_read,video_reader],
    system_prompt="""あなたは次のことができるマルチモーダルアシスタントです：
    - 画像を読み取り、分析する
    - ドキュメント（PDF、CSV、DOCXなど）を処理する
    - 複雑なタスクに高度な推論を使用する。
    - 動画を分析し、詳細な洞察を提供する。
    """
)

print("✅ 組み込みツールでマルチモーダルエージェントを作成しました！")

In [ ]:
# 例4. ツールの直接使用
video_analysis = multimodal_agent.tool.video_reader_local(
     video_path="data-sample/moderation-video.mp4", 
     text_prompt="この動画の主な要素は何ですか？"
)

In [ ]:
print(video_analysis)

## まとめ

このノートブックでは、以下を学習しました：

✅ `@tool`デコレータでカスタムツールを作成する方法

✅ エージェントにツールを追加する方法

✅ strands_toolsから組み込みツールを使用する方法

✅ 複数のツールでエージェントを作成する方法

✅ ツールを直接呼び出す方法


### 次のステップ

次のノートブックに進んで、Model Context Protocol（MCP）統合について学習しましょう！